In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1

import pandas as pd
import os

file_path = os.path.join(path, "Q3_data.csv")  # 'path' from kagglehub cell
df = pd.read_csv(file_path)


In [ ]:
# Task 2
df.head()

In [ ]:
# Task 3
df.info()

In [ ]:
# Task 4
df.describe()

In [ ]:
# Task 1
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(exclude=["int64", "float64"]).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# Task 2
print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after:", df.duplicated().sum())


In [ ]:
# Task 3
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))


In [ ]:
# Task 4
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [ ]:
# Task 5
target_col = "target"  # change if your target column has a different name
print(df[target_col].value_counts(normalize=True))
print(df[target_col].value_counts())


In [ ]:
# Task 1
X = df.drop(columns=[target_col])
y = df[target_col]


In [ ]:
# Task 2,3,4,5

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from catboost import CatBoostClassifier
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []
acc_scores = []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        verbose=0,
        random_seed=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    f1 = f1_score(y_val, y_pred)
    acc = accuracy_score(y_val, y_pred)

    f1_scores.append(f1)
 acc_scores.append(acc)

print("Average F1 Score across folds:", np.mean(f1_scores))
print("Average Accuracy across folds:", np.mean(acc_scores))


In [ ]:
# Task 1
import matplotlib.pyplot as plt
import numpy as np

# Train once on full data to get feature importances
final_model = CatBoostClassifier(verbose=0, random_seed=42)
final_model.fit(X, y)

importances = final_model.get_feature_importance()
feature_names = X.columns

sorted_idx = np.argsort(importances)[::-1]
sorted_importances = importances[sorted_idx]
sorted_features = feature_names[sorted_idx]

plt.figure(figsize=(10, 6))
plt.bar(range(len(sorted_importances)), sorted_importances)
plt.xticks(range(len(sorted_importances)), sorted_features, rotation=90)
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Task 2
golden_feature = sorted_features[0]
print("Golden feature:", golden_feature)

In [ ]:
# Task Bonus
X_golden = X[[golden_feature]]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores_golden = []
acc_scores_golden = []

for train_idx, val_idx in skf.split(X_golden, y):
    X_train, X_val = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        verbose=0,
        random_seed=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    f1_scores_golden.append(f1_score(y_val, y_pred))
    acc_scores_golden.append(accuracy_score(y_val, y_pred))

print("Golden feature only - Average F1:", np.mean(f1_scores_golden))
print("Golden feature only - Average Accuracy:", np.mean(acc_scores_golden))